# Partie I — MLP et ingénierie PyTorch
### Classification supervisée sur données tabulaires réelles (Breast Cancer Wisconsin)

**Étudiant·e :** Malak — EMSI, promotion 2025/2026
**Module :** Deep Learning — Partie I (30 points)

Ce notebook couvre, dans l'ordre du cahier des charges :
1. Concepts fondamentaux PyTorch (`nn.Module`, paramètres, gradient, `state_dict`, device, forward/backward)
2. Préparation des données (nettoyage, encodage, normalisation, split train/val/test)
3. Deux implémentations du MLP (`nn.Sequential` vs classe personnalisée)
4. Inspection des paramètres (`named_parameters()`, `state_dict()`)
5. Trois stratégies d'initialisation (gaussienne, constante, Xavier)
6. Sauvegarde / rechargement du meilleur modèle
7. Vérification device CPU/GPU
8. Évaluation (accuracy, precision, recall, F1, matrice de confusion)
9. Analyse critique et question de synthèse

> **Dataset :** Breast Cancer Wisconsin (diagnostic), intégré nativement à `scikit-learn`
> (569 patientes, 30 variables cliniques mesurées sur des noyaux cellulaires, cible binaire
> malin/bénin). C'est un des datasets explicitement proposés par le cahier des charges.

**À faire avant de commencer :** `Exécution > Tout exécuter` dans Colab. Le notebook tourne
de bout en bout, télécharge/instancie tout lui-même, aucune dépendance externe en dehors
de `torch`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn` (tous préinstallés sur Colab).

In [ ]:
# 0. Imports et configuration générale
import copy
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

print("Torch version :", torch.__version__)


## 1. Concepts fondamentaux PyTorch

- **`nn.Module`** : classe de base de tout composant différentiable (couche, bloc, modèle
  complet). Elle gère automatiquement l'enregistrement des sous-modules et des paramètres,
  et impose l'implémentation de `forward()` (la propagation avant).
- **Paramètres (`nn.Parameter`)** : tenseurs *entraînables* (poids, biais). Chaque paramètre
  porte un attribut `.requires_grad = True` qui dit à l'autograd de calculer son gradient.
- **Gradient** : dérivée de la fonction de perte par rapport à chaque paramètre, calculée par
  rétropropagation (`loss.backward()`), accumulée dans `param.grad`. L'optimiseur (`optim.Adam`,
  etc.) utilise `param.grad` pour mettre à jour `param.data`.
- **`state_dict()`** : dictionnaire ordonné `{nom_paramètre: tenseur}` représentant *tout
  l'état apprenable* du modèle (poids + biais, et éventuellement buffers comme les moyennes
  de BatchNorm). C'est le format utilisé pour la sauvegarde/rechargement.
- **`device`** : objet PyTorch (`cpu` ou `cuda`) qui indique où vivent les tenseurs en
  mémoire. Modèle et données doivent être sur le **même** device, sinon erreur runtime.
- **Propagation avant (forward)** : calcul de la sortie du réseau à partir de l'entrée,
  couche par couche : `z = W·x + b`, puis fonction d'activation non-linéaire.
- **Rétropropagation (backward)** : application de la règle de la chaîne pour calculer
  `∂Loss/∂W` et `∂Loss/∂b` de la dernière couche vers la première, en réutilisant les
  calculs intermédiaires du forward (algorithme de programmation dynamique).

In [ ]:
# 1.bis Détection du device disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)
if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))


## 2. Préparation des données

Étapes : chargement, exploration rapide (valeurs manquantes, équilibre des classes),
séparation train/val/test (60/20/20, stratifiée), puis normalisation (`StandardScaler`
**ajusté uniquement sur le train**, pour éviter toute fuite d'information).

In [ ]:
# 2.1 Chargement et exploration
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

print("Forme de X :", X.shape)
print("Classes :", dict(zip(*np.unique(y, return_counts=True))))
print("Valeurs manquantes :", X.isna().sum().sum())
X.describe().T.head(10)


In [ ]:
# 2.2 Visualisation de l'équilibre des classes
fig, ax = plt.subplots()
y.value_counts().sort_index().plot(kind="bar", ax=ax, color=["#c0392b", "#27ae60"])
ax.set_xticklabels(["malignant (0)", "benign (1)"], rotation=0)
ax.set_title("Distribution des classes — Breast Cancer Wisconsin")
plt.show()


In [ ]:
# 2.3 Split train / val / test (stratifié pour préserver les proportions de classes)
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values, y.values, test_size=0.4, stratify=y.values, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print(f"Train : {X_train.shape[0]} | Val : {X_val.shape[0]} | Test : {X_test.shape[0]}")

# 2.4 Normalisation — fit UNIQUEMENT sur le train, transform partout ailleurs
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("Dimension d'entrée :", input_dim)


In [ ]:
# 2.5 Conversion en tenseurs PyTorch + DataLoaders
def to_tensor_dataset(X_arr, y_arr):
    X_t = torch.tensor(X_arr, dtype=torch.float32)
    y_t = torch.tensor(y_arr, dtype=torch.float32).unsqueeze(1)  # (N, 1) pour BCEWithLogitsLoss
    return TensorDataset(X_t, y_t)

train_ds = to_tensor_dataset(X_train_s, y_train)
val_ds = to_tensor_dataset(X_val_s, y_val)
test_ds = to_tensor_dataset(X_test_s, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb, yb = next(iter(train_loader))
print("Batch X :", xb.shape, "| Batch y :", yb.shape)


## 3. Deux implémentations du MLP

- **Version A — `nn.Sequential`** : empilement déclaratif de couches, rapide à écrire, mais
  rigide (pas de logique conditionnelle dans le forward, pas de branches multiples).
- **Version B — classe personnalisée (`nn.Module`)** : on définit `__init__` (déclaration des
  couches) et `forward` (logique explicite). Plus verbeux mais flexible — utile dès qu'on a
  des skip-connections, plusieurs sorties, ou du contrôle conditionnel.

Les deux versions ci-dessous ont **strictement la même architecture** :
`30 → 64 → 32 → 1` avec ReLU entre les couches cachées. La sortie est **logit brut** (pas de
sigmoïde appliquée dans le modèle) car on utilise `BCEWithLogitsLoss`, numériquement plus
stable qu'une `Sigmoid` + `BCELoss` séparées.

In [ ]:
# 3.1 Version A — nn.Sequential
def build_mlp_sequential(input_dim, hidden1=64, hidden2=32):
    return nn.Sequential(
        nn.Linear(input_dim, hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, 1),
    )

# 3.2 Version B — classe personnalisée
class MLPCustom(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # logit brut, pas d'activation finale
        return x

mlp_seq = build_mlp_sequential(input_dim).to(device)
mlp_custom = MLPCustom(input_dim).to(device)

print(mlp_seq)
print()
print(mlp_custom)


In [ ]:
# 3.3 Vérification : les deux architectures ont bien le même nombre de paramètres
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Paramètres nn.Sequential :", count_params(mlp_seq))
print("Paramètres classe perso  :", count_params(mlp_custom))
assert count_params(mlp_seq) == count_params(mlp_custom), "Les deux architectures devraient avoir le même nombre de paramètres"


## 4. Inspection des paramètres : `named_parameters()` et `state_dict()`

- `named_parameters()` renvoie un **itérateur** `(nom, Parameter)` — utile pour parcourir et
  modifier les poids en boucle (ex. geler des couches, appliquer une init custom).
- `state_dict()` renvoie un **dictionnaire** `{nom: Tensor}` détaché du graphe de calcul —
  c'est le format de sérialisation utilisé par `torch.save` / `torch.load`.

In [ ]:
# 4.1 named_parameters() — nom, forme, nombre d'éléments, requires_grad
print(f"{'Nom':30s}{'Forme':20s}{'#Eléments':>12s}{'requires_grad':>15s}")
print("-" * 77)
for name, param in mlp_custom.named_parameters():
    print(f"{name:30s}{str(tuple(param.shape)):20s}{param.numel():12d}{str(param.requires_grad):>15s}")


In [ ]:
# 4.2 state_dict() — dictionnaire des tenseurs (sans le graphe d'autograd)
sd = mlp_custom.state_dict()
print("Clés du state_dict :", list(sd.keys()))
print()
print("Exemple — biais de fc3 (couche de sortie) :", sd["fc3.bias"])
print("Type de chaque valeur :", type(sd["fc1.weight"]), "| requires_grad ?", sd["fc1.weight"].requires_grad)


## 5. Boucle d'entraînement générique

Fonction réutilisée pour toutes les expériences (initialisation, comparaison de configs) :
entraîne `n_epochs`, évalue à chaque epoch sur le set de validation, renvoie l'historique
des pertes ainsi qu'une copie du meilleur modèle (selon la perte de validation).

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Une passe complète sur `loader`. Si optimizer est fourni -> mode entraînement."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            n_samples += xb.size(0)
    return total_loss / n_samples


def train_model(model, train_loader, val_loader, n_epochs=60, lr=1e-3, weight_decay=0.0,
                 grad_clip=None, verbose=False):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, n_epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_epoch(model, val_loader, criterion, optimizer=None)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    return history, best_state, best_val_loss


## 6. Trois stratégies d'initialisation : gaussienne, constante, Xavier

On applique chaque stratégie via `model.apply(init_fn)`, qui parcourt récursivement tous
les sous-modules. On entraîne ensuite la **même architecture** (`MLPCustom`) trois fois,
seule l'initialisation change, et on compare la vitesse/qualité de convergence.

- **Gaussienne** `N(0, 0.01)` : init "naïve", peut être trop petite (signal qui s'éteint) ou
  mal calibrée selon la profondeur du réseau.
- **Constante** (tous les poids = 0.01) : casse la symétrie *seulement* si la valeur n'est
  pas nulle, mais tous les neurones d'une même couche reçoivent le même gradient → ils
  restent identiques pendant tout l'entraînement (pathologie classique).
- **Xavier/Glorot uniforme** : variance calibrée selon `fan_in`/`fan_out` pour préserver la
  variance du signal à travers les couches — référence pour des activations symétriques
  comme tanh/sigmoid, et un bon point de départ même avec ReLU.

In [ ]:
def init_gaussian(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.01)
        nn.init.zeros_(m.bias)

def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 0.01)
        nn.init.zeros_(m.bias)

def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

init_strategies = {
    "Gaussienne (std=0.01)": init_gaussian,
    "Constante (0.01)": init_constant,
    "Xavier uniforme": init_xavier,
}

init_results = {}
for name, init_fn in init_strategies.items():
    torch.manual_seed(SEED)
    model = MLPCustom(input_dim).to(device)
    model.apply(init_fn)
    history, best_state, best_val = train_model(model, train_loader, val_loader, n_epochs=60, lr=1e-3)
    init_results[name] = {"history": history, "best_val_loss": best_val, "best_state": best_state}
    print(f"{name:25s} -> meilleure val_loss = {best_val:.4f}")


In [ ]:
# 6.1 Comparaison visuelle des courbes de perte selon l'initialisation
fig, ax = plt.subplots(figsize=(8, 5))
for name, res in init_results.items():
    ax.plot(res["history"]["val_loss"], label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss (BCE)")
ax.set_title("Effet de la stratégie d'initialisation sur la convergence")
ax.legend()
plt.show()


**Lecture attendue du graphique** (à confirmer avec tes propres résultats après exécution) :
l'initialisation **constante** stagne généralement très vite à une perte élevée (les neurones
d'une couche restant parfaitement corrélés, le réseau perd l'essentiel de sa capacité), tandis
que **Xavier** et la **gaussienne à faible variance** convergent normalement, Xavier étant
souvent légèrement plus rapide/stable sur les premières epochs.

## 7. Sélection et entraînement du meilleur modèle, sauvegarde / rechargement

In [ ]:
# 7.1 On sélectionne la stratégie d'initialisation gagnante et on ré-entraîne plus longtemps
best_init_name = min(init_results, key=lambda k: init_results[k]["best_val_loss"])
print("Meilleure stratégie d'initialisation :", best_init_name)

torch.manual_seed(SEED)
final_model = MLPCustom(input_dim).to(device)
final_model.apply(init_strategies[best_init_name])

history, best_state, best_val_loss = train_model(
    final_model, train_loader, val_loader, n_epochs=120, lr=1e-3, weight_decay=1e-4, verbose=True
)
final_model.load_state_dict(best_state)  # on recharge le meilleur point rencontré

fig, ax = plt.subplots()
ax.plot(history["train_loss"], label="train")
ax.plot(history["val_loss"], label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE loss")
ax.set_title("Courbe d'apprentissage — modèle final")
ax.legend()
plt.show()


In [ ]:
# 7.2 Sauvegarde du meilleur modèle sur disque
MODEL_PATH = "best_mlp_breast_cancer.pt"
torch.save(final_model.state_dict(), MODEL_PATH)
print("Modèle sauvegardé dans :", MODEL_PATH)

# 7.3 Rechargement dans une INSTANCE FRESH (pour prouver que la sauvegarde est complète)
reloaded_model = MLPCustom(input_dim).to(device)
reloaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
reloaded_model.eval()

# 7.4 Sanity check : les deux modèles doivent produire EXACTEMENT les mêmes sorties
with torch.no_grad():
    xb_check, _ = next(iter(test_loader))
    xb_check = xb_check.to(device)
    out_original = final_model(xb_check)
    out_reloaded = reloaded_model(xb_check)
    max_diff = (out_original - out_reloaded).abs().max().item()

print(f"Différence maximale entre modèle original et rechargé : {max_diff:.2e}")
assert max_diff < 1e-6, "Le rechargement devrait reproduire exactement les mêmes sorties"
print("OK — la sauvegarde/rechargement est cohérente.")


## 8. Vérification de la cohérence device CPU/GPU

In [ ]:
# 8.1 On force explicitement un passage CPU -> device -> CPU pour vérifier qu'il n'y a pas
#     d'erreur silencieuse et que les tenseurs sont bien synchronisés
model_cpu_copy = MLPCustom(input_dim)
model_cpu_copy.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model_cpu_copy.eval()

x_sample_cpu = torch.tensor(X_test_s[:5], dtype=torch.float32)  # reste sur CPU
x_sample_device = x_sample_cpu.to(device)
model_on_device = MLPCustom(input_dim).to(device)
model_on_device.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model_on_device.eval()

with torch.no_grad():
    out_cpu = model_cpu_copy(x_sample_cpu)
    out_device = model_on_device(x_sample_device).to("cpu")

print("Device du modèle 1 :", next(model_cpu_copy.parameters()).device)
print("Device du modèle 2 :", next(model_on_device.parameters()).device)
print("Écart max CPU vs", device, ":", (out_cpu - out_device).abs().max().item())

try:
    # Ceci doit lever une erreur : mélanger un tenseur CPU avec un modèle GPU (si dispo)
    if device.type == "cuda":
        _ = model_on_device(x_sample_cpu)
except RuntimeError as e:
    print("\nErreur attendue si on mélange les devices (preuve de la vérification) :")
    print(str(e)[:200])


## 9. Évaluation finale sur le set de test (jamais vu pendant l'entraînement)

In [ ]:
final_model.eval()
all_preds, all_probs, all_targets = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = final_model(xb)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_targets.extend(yb.numpy().ravel().astype(int))

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds)
rec = recall_score(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print()
print(classification_report(all_targets, all_preds, target_names=["malignant", "benign"]))


In [ ]:
# 9.1 Matrice de confusion
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["malignant", "benign"], yticklabels=["malignant", "benign"], ax=ax)
ax.set_xlabel("Prédiction")
ax.set_ylabel("Vérité terrain")
ax.set_title("Matrice de confusion — Test set")
plt.show()


## 10. Analyse critique *(template — à compléter avec tes chiffres réels après exécution)*

- **Forces observées :** [À compléter — ex. accuracy test obtenue = **___**, convergence
  stable dès l'epoch ___, peu d'écart train/val → pas de surapprentissage marqué]
- **Limites observées :**
  - Sensibilité à l'initialisation (cf. section 6) : l'init constante casse la capacité
    du réseau — preuve que le MLP n'est *pas* robuste "par défaut" à n'importe quel choix
    d'ingénierie, contrairement à une idée reçue.
  - Le MLP ignore toute structure entre les 30 variables (il les traite comme un sac de
    nombres indépendants) ; si certaines features cliniques étaient corrélées de façon
    non-linéaire complexe, un modèle à structure adaptée (arbres, ou feature engineering
    manuel) pourrait égaler ou dépasser le MLP avec moins de données.
  - Dataset relativement petit (569 exemples) et propre (pas de valeurs manquantes) : le
    MLP profite ici d'un cas "facile" ; sa marge d'avantage sur un modèle plus simple
    (régression logistique, random forest) doit être quantifiée, pas supposée.
- **Pistes d'amélioration :** dropout / régularisation L2 plus poussée, validation croisée
  k-fold plutôt qu'un split unique, comparaison explicite avec une régression logistique
  comme baseline.

## 11. Question de synthèse — Partie I *(template à compléter)*

> *Dans quelle mesure un MLP bien paramétré constitue-t-il une solution pertinente pour la
> classification tabulaire sur un dataset réel, et quelles sont ses principales limites au
> regard de la structure statistique des données étudiées ?*

**Plan de réponse à rédiger (4 paragraphes) :**

1. **Théorie** — Un MLP est un approximateur universel de fonctions continues (théorème
   d'approximation universelle), mais cette garantie est asymptotique : rien ne dit qu'un
   réseau de taille raisonnable, entraîné sur peu de données, atteint cette approximation
   en pratique.
2. **Choix méthodologiques** — Normalisation indispensable (les 30 features de Breast
   Cancer ont des échelles très différentes — rayon vs texture vs symétrie), split
   stratifié pour préserver l'équilibre des classes (malin/bénin ≈ 37/63 %), choix
   d'initialisation Xavier justifié théoriquement et confirmé empiriquement (section 6).
3. **Résultats expérimentaux** — *(à insérer : accuracy/F1 obtenus, comparaison des
   stratégies d'initialisation, courbe d'apprentissage)*.
4. **Analyse critique** — Le MLP est pertinent ici parce que les features sont déjà des
   statistiques agrégées et numériques (pas de structure spatiale/séquentielle à exploiter),
   ce qui est précisément le cas d'usage pour lequel un MLP est approprié — contrairement à
   des images ou du texte brut (cf. Parties II et III), où l'absence de prise en compte de
   la structure des données pénaliserait fortement les performances.